# 19 · Typing & Pydantic

**Track 3 begins: production-grade craft.** Dirty input is the norm. **Type hints** document intent; **`dataclass`**
structures records; **pydantic** *validates and coerces* them at the boundary of
your pipeline — turning messy dicts into trustworthy typed objects or clear
errors.

## Type hints in depth

Annotations describe expected types without enforcing them at runtime. Use
`list[int]`, `dict[str, float]`, `str | None` (optional), and `typing` helpers.
They power editors, `mypy`, and pydantic.

In [ ]:
def total_revenue(amounts: list[float], tax: float = 0.0) -> float:
    return round(sum(amounts) * (1 + tax), 2)

print(total_revenue([10.0, 20.5, 3.25], tax=0.1))
print(total_revenue.__annotations__)

def find_email(customer: dict) -> str | None:   # may return None
    return customer.get('email') or None
print(find_email({'email': ''}))

## `dataclass`: structure without validation

`@dataclass` gives you a typed record with `__init__`/`__repr__`/`__eq__` for
free — but it does **not** validate: a wrong type slips right through.

In [ ]:
from dataclasses import dataclass

@dataclass
class OrderDC:
    order_id: int
    amount: float

bad = OrderDC(order_id='not-an-int', amount='oops')   # no error!
print(bad)   # dataclass trusts you

## pydantic: validation + coercion at the boundary

A pydantic `BaseModel` checks types and **coerces** where sensible (`'42'` →
`42`), applies constraints, and raises a detailed `ValidationError` on bad
data. This is exactly what you want when ingesting external records.

In [ ]:
from pydantic import BaseModel, field_validator

class Order(BaseModel):
    order_id: int
    amount: float
    status: str

    @field_validator('amount')
    @classmethod
    def non_negative(cls, v):
        if v < 0:
            raise ValueError('amount must be >= 0')
        return v

# strings get coerced to the declared types
o = Order(order_id='1042', amount='89.90', status='completed')
print(o)
print('typed amount:', o.amount, type(o.amount))

## Catching bad records cleanly

In a pipeline you validate each incoming record, routing good ones onward and
bad ones to a dead-letter list with a readable reason — no silent corruption.

In [ ]:
from pydantic import ValidationError

incoming = [
    {'order_id': 1, 'amount': 100, 'status': 'completed'},
    {'order_id': 'x', 'amount': 5, 'status': 'completed'},   # bad id
    {'order_id': 3, 'amount': -9, 'status': 'returned'},     # bad amount
]
good, rejected = [], []
for rec in incoming:
    try:
        good.append(Order(**rec))
    except ValidationError as e:
        rejected.append((rec, e.errors()[0]['msg']))

print('accepted:', len(good))
for rec, why in rejected:
    print('rejected', rec['order_id'], '->', why)

## Nested models & defaults

Models compose, so nested JSON validates in one shot; fields can have defaults
and optional types.

In [ ]:
from pydantic import BaseModel

class Payload(BaseModel):
    path: str
    qty: int = 0                 # default when absent

class Event(BaseModel):
    event_id: int
    event_type: str
    payload: Payload

e = Event(event_id=1, event_type='add_to_cart',
          payload={'path': '/cart', 'qty': '2'})
print(e)
print('nested qty:', e.payload.qty, type(e.payload.qty))

### Recap

Type hints document; `dataclass` structures but does not validate; pydantic
validates and coerces at the ingestion boundary, raising detailed errors you can
route to a dead-letter queue; models nest for JSON. Next: concurrency.